In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Actuals

import pandas as pd
import os

file_path = "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Raw Data/hist04z1.xlsx"

df = pd.read_excel(file_path, sheet_name=0, header=None)

# Find actual header row
header_row = df[df.iloc[:,0].astype(str).str.contains("Department or other unit", na=False)].index[0]

df = pd.read_excel(
    file_path,
    sheet_name=0,
    header=header_row
)

# Rename first column
df.rename(columns={df.columns[0]: "Agency Name"}, inplace=True)

# Remove totals and blanks
df = df[
    (~df["Agency Name"].isna()) &
    (~df["Agency Name"].astype(str).str.contains("Total|Undistributed", case=False, na=False))
]

# Unpivot years
year_cols = [c for c in df.columns if str(c).isdigit()]

df_long = df.melt(
    id_vars=["Agency Name"],
    value_vars=year_cols,
    var_name="Fiscal Year",
    value_name="Amount"
)

df_long["Version"] = "Actual"

# Convert 'Amount' to numeric, coercing errors (like '..........') to NaN
df_long['Amount'] = pd.to_numeric(df_long['Amount'], errors='coerce')

# Drop rows where 'Amount' is NaN (this now includes '..........' values)
df_long.dropna(subset=["Amount"], inplace=True)

df_long.to_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_ACTUALS.csv", index=False)

print(df_long.head())

                                Agency Name Fiscal Year   Amount Version
0                        Legislative Branch        1962    196.0  Actual
1                           Judicial Branch        1962     57.0  Actual
2                 Department of Agriculture        1962   6437.0  Actual
3                    Department of Commerce        1962    215.0  Actual
4  Department of Defense--Military Programs        1962  50111.0  Actual


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
#Budget

import pandas as pd
import os

file_path = "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Raw Data/hist05z2.xlsx"

df = pd.read_excel(file_path, sheet_name=0, header=None)

header_row = df[df.iloc[:,0].astype(str).str.contains("Department or other unit", na=False)].index[0]

df = pd.read_excel(
    file_path,
    sheet_name=0,
    header=header_row
)

df.rename(columns={df.columns[0]: "Agency Name"}, inplace=True)

df = df[
    (~df["Agency Name"].isna()) &
    (~df["Agency Name"].astype(str).str.contains("Total|Undistributed", case=False, na=False))
]

year_cols = [c for c in df.columns if str(c).isdigit()]

df_long = df.melt(
    id_vars=["Agency Name"],
    value_vars=year_cols,
    var_name="Fiscal Year",
    value_name="Amount"
)

df_long["Version"] = "Budget"

# Convert 'Amount' to numeric, coercing errors (like '..........') to NaN
df_long['Amount'] = pd.to_numeric(df_long['Amount'], errors='coerce')

df_long.dropna(subset=["Amount"], inplace=True)

df_long.to_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_BUDGET.csv", index=False)

print(df_long.head())

                                Agency Name Fiscal Year   Amount Version
0                        Legislative Branch        1976    936.0  Budget
1                           Judicial Branch        1976    346.0  Budget
2                 Department of Agriculture        1976  20690.0  Budget
3                    Department of Commerce        1976   1732.0  Budget
4  Department of Defense--Military Programs        1976  95503.0  Budget


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:
#Category

import pandas as pd
import os

file_path = "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Raw Data/hist08z1.xlsx"

df = pd.read_excel(file_path, header=1)

# Clean up column names by stripping whitespace
df.columns = df.columns.str.strip()

# Rename the relevant columns to what df.melt expects
# Based on the df.head() output, these are the columns containing the desired labels in the first few rows (index 0, 1, 2, 3).
# df.columns[0] contains 'Fiscal Year' in df.iloc[0]
# df.columns[2] contains 'Discretionary' in df.iloc[0]
# df.columns[5] contains 'Mandatory and Net Interest' in df.iloc[0]
rename_map = {
    df.columns[0]: "Fiscal Year",
    df.columns[2]: "Discretionary",
    df.columns[5]: "Mandatory" # Renaming 'Unnamed: 5' (which contains 'Mandatory and Net Interest') to 'Mandatory'
}
df.rename(columns=rename_map, inplace=True)

# Drop the first 4 rows which contain header metadata (rows 0, 1, 2, 3) because actual data starts from row 4 (index 4)
df = df.iloc[4:].copy()

# Convert the 'Fiscal Year' column to numeric, as it's now the actual year data
df['Fiscal Year'] = pd.to_numeric(df['Fiscal Year'], errors='coerce')
df.dropna(subset=['Fiscal Year'], inplace=True) # Drop rows where Fiscal Year couldn't be converted

df_long = df.melt(
    id_vars=["Fiscal Year"],
    value_vars=["Discretionary", "Mandatory"],
    var_name="Category",
    value_name="Amount"
)
df_long['Amount'] = pd.to_numeric(df_long['Amount'], errors='coerce')

df_long.dropna(subset=["Amount"], inplace=True)

# Create 'Staged Data' directory if it doesn't exist
os.makedirs('Staged Data', exist_ok=True)

df_long.to_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_CATEGORY.csv", index=False)

print(df_long.head())

   Fiscal Year       Category  Amount
0       1962.0  Discretionary    72.1
1       1963.0  Discretionary    75.3
2       1964.0  Discretionary    79.1
3       1965.0  Discretionary    77.8
4       1966.0  Discretionary    90.1


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [5]:
#Account Hierarchy

import pandas as pd
import os

file_path = "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Raw Data/hist03z2.xlsx"

df = pd.read_excel(file_path, header=None)

# Find the row that contains 'Function and Subfunction' which appears to be the actual header for the main data column
header_row_index = df[df.iloc[:, 0].astype(str).str.contains("Function and Subfunction", na=False)].index[0]

# Set the column headers from the identified header row and skip rows before it
df.columns = df.iloc[header_row_index]
df = df[header_row_index+1:].copy() # Use .copy() to avoid SettingWithCopyWarning

# Clean up column names by stripping whitespace
df.columns = df.columns.str.strip()

# Rename the 'Function and Subfunction' column to 'Bureau Name'
df.rename(columns={
    "Function and Subfunction": "Bureau Name"
}, inplace=True)

# Drop rows where 'Bureau Name' is NaN, as these might be empty rows or metadata
df.dropna(subset=["Bureau Name"], inplace=True)

# Create 'Account Name' by identifying subfunction entries
# Subfunctions appear to be indented or have specific patterns. For this file, it seems like
# higher-level functions do not have leading numbers (e.g., 'National Defense'), while subfunctions do (e.g., '051 Department of Defense-Military:').
# Also, some rows are just parent categories (e.g., '050 National Defense:'), and others are sub-categories (e.g., '051 Department of Defense-Military:').
# We will assume that 'Account Name' entries are those that start with digits (like '051'), and 'Bureau Name' are the higher level.

df["Account Name"] = None

# Fill 'Account Name' for rows that appear to be sub-functions
# A simple heuristic: if a 'Bureau Name' entry starts with a digit, it might be an 'Account Name'
# and its 'Bureau Name' would be the preceding non-digit starting entry.
# Let's refine: identify rows that are 'Bureau Name' vs 'Account Name' based on content.

def is_account_name(text):
    return text.strip().startswith(('0', '1', '2', '3', '4', '5', '6', '7', '8', '9'))

# Initialize 'Account Name' with empty string or NaN
df['Account Name'] = df['Bureau Name'].apply(lambda x: x if is_account_name(str(x)) else None)

# Update 'Bureau Name' to be the higher-level function
# This requires iterating or using forward fill logic carefully
current_bureau = None
for index, row in df.iterrows():
    if not is_account_name(str(row['Bureau Name'])):
        current_bureau = row['Bureau Name']
        df.loc[index, 'Account Name'] = None # This row is a Bureau, not an account
    else:
        # This row is an Account, so its Bureau is the last seen current_bureau
        df.loc[index, 'Bureau Name'] = current_bureau

# Drop rows where 'Account Name' is None (these are the top-level 'Bureau Name' rows without a specific account)
df.dropna(subset=['Account Name'], inplace=True)

# Strip leading digits and colons from 'Account Name' for cleaner names if needed
df['Account Name'] = df['Account Name'].astype(str).str.replace(r'^\d+\s*', '', regex=True).str.strip()

# Drop columns that are entirely NaN (e.g., empty year columns in the header)
df.dropna(axis=1, how='all', inplace=True)

# Select only the desired columns (Bureau Name, Account Name, and any year columns for amount)
year_columns = [col for col in df.columns if str(col).isdigit()]
final_columns = ["Bureau Name", "Account Name"] + year_columns
df = df[final_columns].copy()

# Convert year columns to numeric, coercing errors to NaN, then drop NaNs
for col in year_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df.dropna(subset=year_columns, how='all', inplace=True)

# If the goal is a long format, melt the dataframe
df_long = df.melt(
    id_vars=["Bureau Name", "Account Name"],
    value_vars=year_columns,
    var_name="Fiscal Year",
    value_name="Amount"
)

# Convert 'Fiscal Year' to integer
df_long['Fiscal Year'] = pd.to_numeric(df_long['Fiscal Year'], errors='coerce').astype('Int64')

# Drop rows where 'Amount' is NaN (this will remove non-numeric values like '..........')
df_long['Amount'] = pd.to_numeric(df_long['Amount'], errors='coerce')
df_long.dropna(subset=["Amount"], inplace=True)

# Create 'Staged Data' directory if it doesn't exist
os.makedirs('Staged Data', exist_ok=True)

df_long.to_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_ACCOUNT.csv", index=False)

print(df_long.head())

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


               Bureau Name                                       Account Name  \
0                    Other           Subtotal, Department of Defense-Military   
1                    Other                   Atomic energy defense activities   
2                    Other                         Defense-related activities   
3  Total, National Defense  International development and humanitarian ass...   
4  Total, National Defense                  International security assistance   

   Fiscal Year   Amount  
0         1962  50111.0  
1         1962   2074.0  
2         1962    160.0  
3         1962   2883.0  
4         1962   1958.0  


In [34]:
import pandas as pd

acc = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_ACCOUNT.csv")

# Absolute values for allocation basis
acc["AmountAbs"] = acc["Amount"].abs()

acc["BureauTotal"] = (
    acc.groupby(
        ["Fiscal Year", "Bureau Name"]
    )["AmountAbs"]
    .transform("sum")
)

acc["Weight"] = (
    acc["AmountAbs"] /
    acc["BureauTotal"]
)

weights = acc[
    [
        "Fiscal Year",
        "Bureau Name",
        "Account Name",
        "Weight"
    ]
]

weights.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Mapping/FUNCTION_WEIGHTS.csv",
    index=False
)

In [8]:
import pandas as pd

actuals = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_ACTUALS.csv")

agencies = (
    actuals["Agency Name"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print(agencies.tolist())

agencies.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Mapping/UNIQUE_AGENCIES.csv",
    index=False
)

['(Off-budget)', '(On-budget)', 'Corps of Engineers--Civil Works', 'Department of Agriculture', 'Department of Commerce', 'Department of Defense--Military Programs', 'Department of Education', 'Department of Energy', 'Department of Health and Human Services', 'Department of Homeland Security', 'Department of Housing and Urban Development', 'Department of Justice', 'Department of Labor', 'Department of State', 'Department of Transportation', 'Department of Veterans Affairs', 'Department of the Interior', 'Department of the Treasury', 'Environmental Protection Agency', 'Executive Office of the President', 'General Services Administration', 'International Assistance Programs', 'Judicial Branch', 'Legislative Branch', 'National Aeronautics and Space Administration', 'National Science Foundation', 'Office of Personnel Management', 'Other Defense--Civil Programs', 'Other Independent Agencies (Off-Budget)', 'Other Independent Agencies (On-Budget)', 'Small Business Administration', 'Social Sec

In [9]:
accounts = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_ACCOUNT.csv")

bureaus = (
    accounts["Bureau Name"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print(bureaus.tolist())

bureaus.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Mapping/UNIQUE_BUREAUS.csv",
    index=False
)

['(Off-budget)', 'Other', 'Total, Administration of Justice', 'Total, Agriculture', 'Total, Allowances', 'Total, Community and Regional Development', 'Total, Education, Training, Employment, and Social Services', 'Total, Energy', 'Total, General Government', 'Total, General Science, Space, and Technology', 'Total, Health', 'Total, Income Security', 'Total, International Affairs', 'Total, National Defense', 'Total, Natural Resources and Environment', 'Total, Transportation', 'Total, Veterans Benefits and Services']


In [29]:
import pandas as pd

mapping_rules = {
    "Department of Agriculture": "Total, Agriculture",
    "Department of Commerce": "Total, General Government",
    "Department of Defense--Military Programs": "Total, National Defense",
    "Department of Education": "Total, Education, Training, Employment, and Social Services",
    "Department of Energy": "Total, Energy",
    "Department of Health and Human Services": "Total, Health",
    "Department of Homeland Security": "Total, Administration of Justice",
    "Department of Housing and Urban Development": "Total, Community and Regional Development",
    "Department of Justice": "Total, Administration of Justice",
    "Department of Labor": "Total, Education, Training, Employment, and Social Services",
    "Department of State": "Total, International Affairs",
    "Department of Transportation": "Total, Transportation",
    "Department of Veterans Affairs": "Total, Veterans Benefits and Services",
    "Department of the Interior": "Total, Natural Resources and Environment",
    "Department of the Treasury": "Total, General Government",
    "Environmental Protection Agency": "Total, Natural Resources and Environment",
    "Executive Office of the President": "Total, General Government",
    "General Services Administration": "Total, General Government",
    "International Assistance Programs": "Total, International Affairs",
    "Judicial Branch": "Total, Administration of Justice",
    "Legislative Branch": "Total, General Government",
    "National Aeronautics and Space Administration": "Total, General Science, Space, and Technology",
    "National Science Foundation": "Total, General Science, Space, and Technology",
    "Office of Personnel Management": "Total, General Government",
    "Other Defense--Civil Programs": "Total, National Defense",
    "Small Business Administration": "Total, General Government",
    "Social Security Administration (Off-Budget)": "Total, Income Security",
    "Social Security Administration (On-Budget)": "Total, Income Security",
    "Other Independent Agencies (Off-Budget)": "Total, General Government",
    "Other Independent Agencies (On-Budget)": "Total, General Government",
    "(On-budget)": "Total, General Government",
    "(Off-budget)": "Total, General Government",
    "Corps of Engineers--Civil Works": "Total, Transportation"
}

mapping_df = pd.DataFrame(
    list(mapping_rules.items()),
    columns=["Agency Name", "Bureau Name"]
)

mapping_df.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Mapping/AGENCY_BUREAU_MAP.csv",
    index=False
)

print(mapping_df.head())

                                Agency Name  \
0                 Department of Agriculture   
1                    Department of Commerce   
2  Department of Defense--Military Programs   
3                   Department of Education   
4                      Department of Energy   

                                         Bureau Name  
0                                 Total, Agriculture  
1                          Total, General Government  
2                            Total, National Defense  
3  Total, Education, Training, Employment, and So...  
4                                      Total, Energy  


In [11]:
import pandas as pd

actuals = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_ACTUALS.csv"
)

weights = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Mapping/FUNCTION_WEIGHTS.csv"
)

# Load the mapping rules from the CSV file saved previously
mapping_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Mapping/AGENCY_BUREAU_MAP.csv")
mapping_rules = mapping_df.set_index("Agency Name")["Bureau Name"].to_dict()

actuals["Bureau Name"] = (
    actuals["Agency Name"]
    .map(mapping_rules)
)

actuals = actuals.merge(
    weights,
    on=["Fiscal Year","Bureau Name"],
    how="left"
)

actuals["Amount"] = (
    actuals["Amount"] *
    actuals["Weight"]
)

actuals["Version"] = "Actual"

actuals_final = actuals[
    [
        "Fiscal Year",
        "Agency Name",
        "Bureau Name",
        "Account Name",
        "Version",
        "Amount"
    ]
]

actuals_final.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Business Data/ACTUALS_ENRICHED.csv",
    index=False
)

print(actuals_final.shape)
print(actuals_final.head())

(8345, 6)
   Fiscal Year         Agency Name                       Bureau Name  \
0         1962  Legislative Branch         Total, General Government   
1         1962  Legislative Branch         Total, General Government   
2         1962  Legislative Branch         Total, General Government   
3         1962  Legislative Branch         Total, General Government   
4         1962     Judicial Branch  Total, Administration of Justice   

                                   Account Name Version      Amount  
0  Interest on Treasury debt securities (gross)  Actual  157.490749  
1    Interest received by on-budget trust funds  Actual   14.229427  
2   Interest received by off-budget trust funds  Actual   10.516652  
3                                Other interest  Actual   13.763172  
4                         Legislative functions  Actual    6.164034  


In [12]:
stg_actuals = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_ACTUALS.csv"
)

print("Source Total")
print(stg_actuals["Amount"].sum())

print("Allocated Total")
print(actuals_final["Amount"].sum())

Source Total
129606935.0
Allocated Total
129606934.99999997


In [13]:
import pandas as pd

budget = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_BUDGET.csv"
)

weights = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Mapping/FUNCTION_WEIGHTS.csv"
)

mapping_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Mapping/AGENCY_BUREAU_MAP.csv")
mapping_rules = mapping_df.set_index("Agency Name")["Bureau Name"].to_dict()

budget["Bureau Name"] = (
    budget["Agency Name"]
    .map(mapping_rules)
)

budget = budget.merge(
    weights,
    on=["Fiscal Year","Bureau Name"],
    how="left"
)

budget["Amount"] = (
    budget["Amount"] *
    budget["Weight"]
)

budget["Version"] = "Budget"

budget_final = budget[
    [
        "Fiscal Year",
        "Agency Name",
        "Bureau Name",
        "Account Name",
        "Version",
        "Amount"
    ]
]

budget_final.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Business Data/BUDGET_ENRICHED.csv",
    index=False
)

print(budget_final.shape)

(6564, 6)


In [15]:
category_map = {
    "Total, National Defense": "Discretionary",
    "Total, Transportation": "Discretionary",
    "Total, Agriculture": "Discretionary",
    "Total, Energy": "Discretionary",
    "Total, Health": "Mandatory",
    "Total, Income Security": "Mandatory",
    "Total, Veterans Benefits and Services": "Mandatory",
    "Total, General Government": "Discretionary",
    "Total, Administration of Justice": "Discretionary",
    "Total, International Affairs": "Discretionary",
    "Total, Natural Resources and Environment": "Discretionary",
    "Total, Community and Regional Development": "Discretionary",
    "Total, General Science, Space, and Technology": "Discretionary",
    "Total, Education, Training, Employment, and Social Services": "Discretionary",
    "Other": "Discretionary"
}

actuals_final["Category"] = actuals_final["Bureau Name"].map(category_map)
budget_final["Category"] = budget_final["Bureau Name"].map(category_map)

In [16]:
import pandas as pd

final = pd.concat(
    [actuals_final, budget_final],
    ignore_index=True
)

final = final[
    [
        "Fiscal Year",
        "Agency Name",
        "Bureau Name",
        "Account Name",
        "Category",
        "Version",
        "Amount"
    ]
]

final.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/Colab Projects/Business Data/FPA_FINAL.csv",
    index=False
)

print(final.shape)
print(final.head())

(14909, 7)
   Fiscal Year         Agency Name                       Bureau Name  \
0         1962  Legislative Branch         Total, General Government   
1         1962  Legislative Branch         Total, General Government   
2         1962  Legislative Branch         Total, General Government   
3         1962  Legislative Branch         Total, General Government   
4         1962     Judicial Branch  Total, Administration of Justice   

                                   Account Name       Category Version  \
0  Interest on Treasury debt securities (gross)  Discretionary  Actual   
1    Interest received by on-budget trust funds  Discretionary  Actual   
2   Interest received by off-budget trust funds  Discretionary  Actual   
3                                Other interest  Discretionary  Actual   
4                         Legislative functions  Discretionary  Actual   

       Amount  
0  157.490749  
1   14.229427  
2   10.516652  
3   13.763172  
4    6.164034  


In [17]:
import pandas as pd

final = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Business Data/FPA_FINAL.csv")

print(final.groupby("Version")["Amount"].sum())

stg_actuals = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_ACTUALS.csv")
stg_budget = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Colab Projects/Staged Data/STG_BUDGET.csv")

print(stg_actuals["Amount"].sum())
print(stg_budget["Amount"].sum())

Version
Actual    129606935.0
Budget    134091015.0
Name: Amount, dtype: float64
129606935.0
134091015.0
